# **PART II: RAG SYSTEM**

Group Members:

-Casucci Leonardo, student ID: 2196383
  
  -Frigo Gianmaria,  student ID: 2196376

Master Program: Computer Engineering

[Short Description and project goal]

# **DATASET**

[introduction + summary of your dataset through descriptive statistics]

## Dataset Loading

In [249]:
#extraction of the data

# Install the library to handle massive document sets
#!pip install -q datasets langchain_text_splitters

from datasets import load_dataset

dataset = load_dataset(
    "KillerShoaib/Jeffrey-Epstein-Emails-From-Epstein-Files"
)

df = dataset["train"].to_pandas()
df.head()

,doc_id,subject,preview,from_name,from_email,to,date,body
0,4accfb5f3ed84656e9762740081a4579,cody rudland: You are dead,- Lol good riddance,cody rudland,[redacted],<jeeproject@yahoo.com>,"Aug 13, 2019 6:48 AM",Lol good riddance
1,97d4a52d1df3948368770068262d2aab,Cecilia Steen: With love,"- My dearest Jeffrey, I don't know when you'l...",Cecilia Steen,[redacted],"JE project <jeeproject@yahoo.com>, Jeffrey Eps...","Jul 25, 2019 9:16 AM","My dearest Jeffrey,\nI don't know when you'll ..."
2,HOUSE_OVERSIGHT_016203,Flipboard 10 for Today: 11 questions for Mueller,- Flipboard Interesting stories worth your ti...,Flipboard 10 for Today,editorialstaff@flipboard.com,jeevacation@gmail.com,"Jul 13, 2019 9:06 PM",Flipboard\nInteresting stories worth your time...
3,HOUSE_OVERSIGHT_028801,"Flipboard Week in Review: Alex Acosta resigns,...",- Flipboard Biggest news stories from the pa...,Flipboard Week in Review,editorialstaff@flipboard.com,jeevacation@gmail.com,"Jul 13, 2019 6:48 PM",Flipboard\nBiggest news stories from the past ...
4,HOUSE_OVERSIGHT_029504,[redacted]: [redacted],"- Dear Jeff, I read the news papers today, I ...",[redacted],[redacted],Jeffrey Epstein <jeevacation@gmail.com>,"Jul 10, 2019 8:07 AM","Dear Jeff,\nI read the news papers today, I ca..."


## Dataset Profiling

In [250]:
df.shape
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 14835 entries, 0 to 14834
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   doc_id      14835 non-null  str  
 1   subject     14835 non-null  str  
 2   preview     14835 non-null  str  
 3   from_name   14835 non-null  str  
 4   from_email  14835 non-null  str  
 5   to          14835 non-null  str  
 6   date        14835 non-null  str  
 7   body        14835 non-null  str  
dtypes: str(8)
memory usage: 19.8 MB


doc_id        0
subject       0
preview       0
from_name     0
from_email    0
to            0
date          0
body          0
dtype: int64

In [251]:
print(f"Number of unique senders: {df['from_email'].nunique()}")
print(f"Number of unique recipients: {df['to'].nunique()}")

Number of unique senders: 455
Number of unique recipients: 1434


In [252]:
df["body"].str.len().describe()

count     14835.000000
mean       1121.424065
std        6707.694938
min           1.000000
25%          84.000000
50%         377.000000
75%         893.000000
max      354403.000000
Name: body, dtype: float64

In [253]:
df["subject"].value_counts().head(10)

subject
jeffrey E.: Re:                                                                                                                                                 1685
J. Epstein: Re:                                                                                                                                                  519
J: Re:                                                                                                                                                           278
J. Epstein: (no subject)                                                                                                                                         220
jeffrey E.: Re: Fwd:                                                                                                                                             217
jeffrey E.: (no subject)                                                                                                                                         198
Je

## Dataset Clean Up

#### Checking for near-empty bodies. They are all reasonable, so we keep them.

In [254]:
df["body_len"] = df["body"].str.len()

short_bodies = df[df["body_len"] < 20]
df.sort_values("body_len")[["doc_id", "subject", "body", "body_len"]].head(20)

,doc_id,subject,body,body_len
459,HOUSE_OVERSIGHT_028576,J: RE: Re:,?,1
1324,HOUSE_OVERSIGHT_032788,jeffrey E.: Re:,?,1
6290,4c8ebdda8fbdccef86a17cee3e82562d,jeffrey E.: Re:,3,1
2169,HOUSE_OVERSIGHT_032706,Kathy Ruemmler: Re:,?,1
5894,HOUSE_OVERSIGHT_029831,[redacted]: Re: Reuters / lawsuit against Jeff...,j,1
501,HOUSE_OVERSIGHT_028589,J: Re:,?,1
1334,HOUSE_OVERSIGHT_032785,jeffrey E.: Re: [redacted],?,1
568,HOUSE_OVERSIGHT_028611,J: Re:,?,1
603,HOUSE_OVERSIGHT_026329,J: Re:,?,1
5757,HOUSE_OVERSIGHT_032873,"Thomas Jr., Landon: Fwd: The book on you...",no,2


#### Normalizing whitespace.

In [255]:
import re

def normalize_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for col in ["subject", "preview", "from_name", "from_email", "to", "date", "body"]:
    df[col] = df[col].apply(normalize_text)

#### Counting recipients

In [256]:
import re

def count_recipients_by_angle_brackets(to_field):
    if not isinstance(to_field, str) or to_field.strip() == "":
        return 0

    return max(1, len(re.findall(r"<[^<>]+>", to_field)))

df["num_recipients"] = df["to"].apply(count_recipients_by_angle_brackets)
df[["to", "num_recipients"]].sort_values("num_recipients", ascending=False).head(20)

,to,num_recipients
8437,valdemar iodice <valdemar@iodice.com.br>cc isa...,67
1617,"Alan Rogers <[redacted]>, Jeffrey Epstein <jee...",34
6388,"Allen West <[redacted]>, Rafael Bardaji <[reda...",31
12509,"Allen West <[redacted]>, Rafael Bardaji <[reda...",31
7016,"Glenn Young <shakescenes@aol.com>, gm2127 <gm2...",20
5748,Gary Savitzky <GarySavitzky@kamillagordon.oran...,20
6580,Gary Savitzky <rafael.prates@prismabrasil.com....,20
6695,"<sstrohaver@nlg.com>, <jboltax@nextjump.co>, <...",20
5519,"Schecter, Daniel (CC) <[redacted]>, Dunlavey, ...",17
8440,<JEEPROJECT@YAHOO.COM>cc <ehanna@nysgmail.com>...,16


#### Row Deduplication.

Checking if dates are formatted in a consistent way before removing duplicates.

In [257]:
import pandas as pd

df["parsed_date"] = pd.to_datetime(df["date"], errors="coerce")

print("Unparseable dates:", df["parsed_date"].isna().sum())
print("Date range:", df["parsed_date"].min(), "to", df["parsed_date"].max())

C:\Users\gianm\AppData\Local\Temp\ipykernel_20408\230258536.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["parsed_date"] = pd.to_datetime(df["date"], errors="coerce")


Unparseable dates: 0
Date range: 1970-01-01 00:00:00 to 2019-08-13 06:48:00


There is a large number of repeated records appears when using a content-based duplicate key

In [258]:
dedup_key = ["subject", "from_email", "to", "num_recipients", "date", "body"]

print("Full-row duplicates:", df.duplicated().sum())
print("Content-based duplicates:", df.duplicated(subset=dedup_key).sum())

Full-row duplicates: 11
Content-based duplicates: 1236


In [259]:
duplicate_groups = (
    df[df.duplicated(subset=dedup_key, keep=False)]
    .groupby(dedup_key)
    .agg(
        n_rows=("doc_id", "size"),
        doc_ids=("doc_id", lambda x: list(x)),
        n_unique_doc_ids=("doc_id", "nunique"),
        n_unique_previews=("preview", "nunique"),
        n_unique_from_names=("from_name", "nunique")
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

duplicate_groups.head(10)

,subject,from_email,to,num_recipients,date,body,n_rows,doc_ids,n_unique_doc_ids,n_unique_previews,n_unique_from_names
594,jeffrey E.: Re:,jeevacation@gmail.com,[redacted],1,"Sep 8, 2017 11:52 AM",what is your schedule in new york how long wil...,11,"[HOUSE_OVERSIGHT_032622, HOUSE_OVERSIGHT_02366...",11,10,1
595,jeffrey E.: Re:,jeevacation@gmail.com,[redacted],1,"Sep 8, 2017 1:33 PM",I will get larson to call you,11,"[HOUSE_OVERSIGHT_032622, HOUSE_OVERSIGHT_02366...",11,10,1
319,"Thomas Jr., Landon: Re: Saudi money",[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Oct 19, 2016 9:41 AM",Interesting. CEO of big finance form told me t...,9,"[HOUSE_OVERSIGHT_031511, HOUSE_OVERSIGHT_03163...",9,9,1
620,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 7:54 AM",Doesn't look like you are prioritizing your sc...,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
619,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 7:15 AM",Most girls do not have to worry about this crap.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
618,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 6:15 AM","Agreed, but I need to be prepared to say yes b...",8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
611,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 3:02 PM","Uh, yeah. We will now go through a period of s...",8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
612,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 3:44 PM",Gov is not jewish,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
609,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 2:57 PM",Reid's guy went down on all 7 counts.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1
610,jeffrey E.: Re: Fwd:,[redacted],jeffrey E. <jeevacation@gmail.com>,1,"Sep 19, 2014 2:59 PM",Yep. And now he is a repeat offender.,8,"[HOUSE_OVERSIGHT_023418, HOUSE_OVERSIGHT_02584...",8,8,1


For the RAG system, repeated identical email content is not useful and may cause the top-k results to contain multiple copies of the same evidence.

The internal document IDs are still useful for source reporting. Therefore, we collapse rows with identical email content into a single row that preserves all `doc_id`s.


In [260]:
df_grouped = (
    df.groupby(dedup_key, dropna=False)
    .agg(
        doc_ids=("doc_id", lambda x: sorted(set(x))),
        n_original_rows=("doc_id", "size"),
        preview=("preview", "first"),
        from_name=("from_name", "first"),
        parsed_date=("parsed_date", "first")
    )
    .reset_index()
)

print("Original rows:", len(df))
print("Rows after content-based grouping:", len(df_grouped))
print("Rows collapsed:", len(df) - len(df_grouped))

Original rows: 14835
Rows after content-based grouping: 13599
Rows collapsed: 1236


## Chunking

In [261]:
def build_header(row):
    # Show up to 5 document IDs to avoid long headers
    doc_ids = ", ".join(row["doc_ids"][:5])
    
    if len(row["doc_ids"]) > 5:
        doc_ids += f", ... ({len(row['doc_ids'])} total)"

    # Truncate recipient list if it's too long
    recipients = str(row["to"])
    if len(recipients) > 200:
        recipients = recipients[:200].rstrip() + " ... (truncated)"

    # Truncate subject if it's too long
    subject = str(row["subject"])
    if len(subject) > 200:
        subject = subject[:200].rstrip() + " ... (truncated)"

    return f"""Subject: {subject}
From: {row['from_name']} <{row['from_email']}>
To: {recipients} of {row['num_recipients']} total recipients
Date: {row['date']}
Source document IDs: {doc_ids}
"""

In [262]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [263]:
chunk_records = []

for row_idx, row in df_grouped.iterrows():
    header = build_header(row)

    body_chunks = splitter.split_text(row["body"])

    for chunk_idx, body_chunk in enumerate(body_chunks):
        chunk_text = f"""{header}

Body chunk {chunk_idx + 1} of {len(body_chunks)}:
{body_chunk}"""

        chunk_records.append({
            "chunk_id": f"{row_idx}_{chunk_idx}",
            "email_id": row_idx,
            "chunk_index": chunk_idx,
            "num_chunks": len(body_chunks),

            # Metadata
            "doc_ids": row["doc_ids"],
            "n_original_rows": row["n_original_rows"],
            "subject": row["subject"],
            "from_name": row["from_name"],
            "from_email": row["from_email"],
            "to": row["to"],
            "date": row["date"],

            # Text used for retrieval and prompt
            "text_for_embedding": chunk_text,
            "text_for_prompt": chunk_text,
        })

chunks_df = pd.DataFrame(chunk_records)

chunks_df.head()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


,chunk_id,email_id,chunk_index,num_chunks,doc_ids,n_original_rows,subject,from_name,from_email,to,date,text_for_embedding,text_for_prompt
0,0_0,0,0,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
1,0_1,0,1,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
2,0_2,0,2,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
3,0_3,0,3,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...
4,0_4,0,4,6,[a9daccc94ec91c8d1dc5f4f769c4d722],1,: !!! Re: Love from Prague,,petranemcova@mac.com,Jeffrey Epstein <jeeproject@yahoo.com>,"Jul 29, 2008 4:34 PM",Subject: : !!! Re: Love from Prague\nFrom: <p...,Subject: : !!! Re: Love from Prague\nFrom: <p...


In [264]:
chunks_len = chunks_df["text_for_embedding"].apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)
longest = chunks_len.max()
longest_chunk = chunks_df.iloc[chunks_len.idxmax()]


print(f"Longest chunk with header: {longest} tokens")
print(f"Text of longest chunk:\n{longest_chunk['text_for_prompt']}")

Longest chunk with header: 693 tokens
Text of longest chunk:
Subject: Kathy Ruemmler: Re: Press: LA Times - Federal jury decides Middle East bank did not defraud Orange County entrepreneur
From: Pizzurro, Frank (LA) <[redacted]>
To: Dunlavey, Dean (OC) <[redacted]>, Schecter, Daniel (CC) <[redacted]>, Mohebbi, Nima (LA) <[redacted]>, Ruemmler, Kathy (DC) <[redacted]>cc #L&W BD PR (US) <#L&WBDPRUS@LW.com>, Wine, Jamie (NY) <[redac ... (truncated) of 13 total recipients
Date: Aug 11, 2016 7:41 PM
Source document IDs: HOUSE_OVERSIGHT_030101


Body chunk 2 of 3:
. The Gulf States rely heavily on migrants to work construction and other low-wage jobs, offering a readymade market for SpanCash. InfoSpan aimed to allow migrants to transfer money back home far more cheaply than Western Union or hawala, a traditional Middle Eastern broker-to-broker money transfer system. A study from McKinsey & Co., cited in court records, projected annual revenue of $3.5 billion by the deal’s fifth year, with In

# **LIBRARIES USED**

[Introduction + documentation of libraries used for sentence embedding, vector store, LLM quantization, etc.]

In [ ]:
#code for setup and initialization of libraries

# **PROMPT ENGINEERING**

[Define function that given a query, retrieves significant documents and crafts the query]

[Discussion of design choices taken + results: first bad option discared because..., second option, ..., final option]

In [ ]:
#code for the prompt crafting

# **TESTING**

[Choose around 10 to 20 passages from your document repository, and for each passage extract question/answer pairs.]

[To evaluate your system on the micro-test set, use the same chatBot that has extracted the question/answer pairs, and prompt it to act as a judge, passing the question, the gold answer, and the answer you obtain from your system, and ask to provide some score in a given range.]

[Analyse possible errors and inaccuracies, and discuss possible ways of improving your system.]

In [ ]:
#code for testing

# **USE OF GENERATIVE AI**

[Describe how AI has been used including examples of used promts]